# C1.6 · Attacking evaluation itself

**Function C — Offensive Security & Research → The Pentester / Red Teamer**  ·  *Security of AI*

---

**Risk.** If the eval can be fooled, the assurance is theatre.

**Control.** Eval gaming, sandbagging, contamination and judge manipulation as test cases.

**This lab.** Game the eval deliberately, then close the hole you used.

| | |
|---|---|
| Open-source tooling | Cyber Commons eval harness |
| Open-weight models | Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("C1.6"))

Attacking evaluation itself. If you can make a harness score well without being good, so can the vendor whose benchmark you are reading.

In [ ]:
from cybercommons import evalkit

# a "harness" that has no capability whatsoever
answers = {f"q{i}": ('{"qid":"q%d","cwe":"CWE-89","file":"CWE-89/1.py",'
                     '"rationale":"user input is concatenated"}' % i)
           for i in range(1, 21)}
g = evalkit.gameable_score(answers)
for k, v in g.items():
    print(f"{k:38s} {v}")

Perfect conformance, and a respectable-looking accuracy purely from guessing the majority class. Now show what a held-out key does to it.

In [ ]:
truths = {f"q{i}": evalkit.Truth(f"q{i}",
              "CWE-89" if i <= 6 else "CWE-78" if i <= 13 else "CWE-22",
              f"CWE-{'89' if i <= 6 else '78' if i <= 13 else '22'}/{i}.py")
          for i in range(1, 21)}
rep = evalkit.evaluate(answers, truths)
print(rep.render())

Conformance stays at 1.0. Expert accuracy collapses, because the held-out key varies the file *and* the class. Any benchmark whose headline number survives this treatment is measuring formatting.

### Expect

The gameability report shows conformance 1.0 with majority-class accuracy around 1.0, while evaluation against the varied held-out key gives conformance 1.0 and a far lower expert accuracy.

### Your turn

Apply this to a published benchmark result you rely on. Ask two questions: what is the class balance, and is the key held out? Most public numbers do not answer either.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/C1.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*